In [ ]:
import itertools
import json

import numpy as np
from scipy.io import loadmat

# File path, type pair, and selected measurement windows (stop is exclusive)
file_path = 'data_simulation/structure_function'
type_pair = ("A", "A")  # Two monomer names, including any underscores
iterations = range(100000, 500000, 100000)
file_names = [f"{file_path}_{step:06d}.mat" for step in iterations]
if not file_names:
    raise ValueError("Select at least one structure-function file.")

# Load the grid and cell from the first selected file: the lattice angles are saved with the
# structure functions, older files keep those of the input in initial_params
geometry = loadmat(file_names[0], squeeze_me=True, variable_names=['nx', 'lx', 'angles', 'initial_params'])
nx = geometry['nx'].astype(int)
lx = np.asarray(geometry['lx'], dtype=float)
angles = np.array([90.0, 90.0, 90.0])
if 'angles' in geometry:
    angles = np.asarray(geometry['angles'], dtype=float)
elif 'initial_params' in geometry and 'angles' in (geometry['initial_params'].dtype.names or ()):
    input_angles = np.asarray(geometry['initial_params']['angles'].item(), dtype=float)
    if input_angles.size == 3:
        angles = input_angles
alpha, beta, gamma = np.deg2rad(angles)

# Lattice vectors as rows (a along x, b in the xy plane, as in ComputationBox) and the
# reciprocal metric G* = (L L^T)^-1: |k|^2 = (2 pi)^2 m^T G* m for the integer index m
cx = np.cos(beta)
cy = (np.cos(alpha) - np.cos(beta)*np.cos(gamma))/np.sin(gamma)
L = np.array([lx[0]*np.array([1.0, 0.0, 0.0]),
              lx[1]*np.array([np.cos(gamma), np.sin(gamma), 0.0]),
              lx[2]*np.array([cx, cy, np.sqrt(1.0 - cx**2 - cy**2)])])
G_star = np.linalg.inv(L @ L.T)

# Weight file means by the actual number of samples in each measurement window
sample_counts = []
for file_name in file_names:
    record = loadmat(file_name, variable_names=['sample_count'])
    if 'sample_count' not in record:
        sample_counts.append(None)
        continue
    count = np.asarray(record['sample_count'])
    if (count.size != 1 or count.dtype.kind not in 'iuf' or not np.isfinite(count).all()
            or count.item() < 0 or count.item() != np.floor(count.item())):
        raise ValueError(f"sample_count must be a finite nonnegative integer in {file_name}.")
    sample_counts.append(float(count.item()))
# Legacy files all receive equal weight; unknown counts cannot be mixed with known counts.
if all(count is None for count in sample_counts):
    sample_counts = [1.0] * len(file_names)
elif any(count is None for count in sample_counts):
    raise ValueError("Cannot mix structure-function files with and without sample_count.")
if not any(count > 0 for count in sample_counts):
    raise ValueError("The selected structure-function files contain no samples.")

# Multiplicity of each kz plane in the full Fourier spectrum
mode_weights = np.full(nx[2]//2 + 1, 2.0)
mode_weights[0] = 1.0
if nx[2] % 2 == 0:
    mode_weights[-1] = 1.0

# Calculate k²/(2π)² and group Fourier modes with equal magnitude. Each stored index stands for
# the alias m + a n with the smallest |k|, as the solvers choose it (docs/theory/NumericalMethods.md,
# Section 3.5). Only an axis with more than one grid point that the metric couples to another axis
# can have it beyond the centred index; |m_d + a_d n_d| <= sqrt(max f (L L^T)_dd) bounds the search
def metric_form(m):
    return sum(G_star[p, q]*m[p]*m[q] for p in range(3) for q in range(3))


m = np.meshgrid(np.arange(nx[0]), np.arange(nx[1]), np.arange(nx[2]//2 + 1), indexing='ij')
m = [np.where(m[d] <= nx[d]//2, m[d], m[d] - nx[d]) for d in range(2)] + [m[2]]
k2 = metric_form(m)
coupled = [np.max(np.abs(np.delete(G_star[d], d))) > 1e-12*np.max(np.abs(G_star)) for d in range(3)]
reach = [int(np.ceil(np.sqrt(k2.max()*(L @ L.T)[d, d])/nx[d])) + 1 if coupled[d] and nx[d] > 1 else 0
         for d in range(3)]
for shift in itertools.product(*[range(-r, r + 1) for r in reach]):
    k2 = np.minimum(k2, metric_form([m[d] + shift[d]*nx[d] for d in range(3)]))

# Shells: values equal to a relative 1e-9 (round-off of the metric), labelled by k²/(2π)²
# rounded to 7 decimals
order = np.argsort(k2, axis=None)
k2_sorted = k2.ravel()[order]
first = np.concatenate(([True], np.diff(k2_sorted) > 1e-9*np.maximum(k2_sorted[1:], 1.0)))
k2_unique = np.round(k2_sorted[first], 7)
k2_mapping = np.empty(k2.size, dtype=int)
k2_mapping[order] = np.cumsum(first) - 1
k2_mapping = k2_mapping.reshape(k2.shape)

sf_mag = np.zeros_like(k2_unique)
sf_count = np.zeros_like(k2_unique)
for file_name, sample_count in zip(file_names, sample_counts):
    if sample_count == 0:
        continue
    print(file_name)
    metadata = loadmat(file_name, squeeze_me=True,
                       variable_names=['pair_key_format_version', 'pair_keys_json'])
    if 'pair_key_format_version' in metadata or 'pair_keys_json' in metadata:
        if metadata.get('pair_key_format_version') != 1 or 'pair_keys_json' not in metadata:
            raise ValueError(f"Invalid pair-name mapping in {file_name}.")
        mapping = json.loads(str(metadata['pair_keys_json']))
        matches = [suffix for suffix, names in mapping.items() if names == sorted(type_pair)]
        if len(matches) != 1:
            raise ValueError(f"Pair {type_pair} has no unique entry in {file_name}.")
        suffix = matches[0]
    else:
        suffix = '_'.join(sorted(type_pair))  # Files written before pair-name metadata
    key = 'structure_function_' + suffix
    data = loadmat(file_name, variable_names=[key])
    v = data[key]
    for i in range(nx[0]):
        for j in range(nx[1]):
            for k in range(nx[2]//2 + 1):
                idx = k2_mapping[i, j, k]
                weight = mode_weights[k] * sample_count
                sf_mag[idx] += v[i, j, k].real * weight
                sf_count[idx] += weight

x = np.sqrt(k2_unique) * 2 * np.pi
y = sf_mag / sf_count
y[0] = 0.0  # Remove the k=0 mode


In [ ]:
import matplotlib.pyplot as plt

# Plot
plt.figure(dpi=100)
plt.style.use('classic')

plt.plot(x, y)
plt.title('Structure function')
plt.xlabel('$kR_0$')
plt.ylabel('$S(k)ρ_0N$')
plt.yscale('log')
plt.xlim([2, 14])
plt.ylim([0.01, 1000])
plt.grid(True)
plt.show()

In [ ]:
import pandas

# Write
df = pandas.DataFrame({'k': x, 's(k)': y})
df.to_excel("data.xlsx", sheet_name='sheet1', index=False)